In [ ]:
%pip install openpyxl scikit-learn


# 1 — Amostragem adaptativa orientada aos rankings de CEQ (estudo de caso 2: caminhões, 5 turmas)

**Objetivo**: separar a base em duas partes **disjuntas**:
- a **amostra** (`amostra.xlsx`), em que o notebook 2 calcula os rankings de atributos;
- a **base restante** (`base_restante.xlsx`), em que os modelos de classificação são treinados e avaliados.

Selecionar atributos e avaliar modelos com linhas diferentes evita o viés de seleção, em que o modelo é avaliado nos mesmos dados que escolheram seus atributos.

**Método**: o mesmo do estudo de caso 1 (SECOM). A única diferença é a estrutura das classes. No SECOM há uma classe de referência (unidades conformes). Aqui há cinco turmas (A a E) e **cada turma é a classe de referência do seu próprio cenário**; por isso a regra de parada é verificada nos cinco cenários ao mesmo tempo.

**Entrada**: planilha `.xlsx` com os rótulos das colunas na 1ª linha e a turma na última coluna (`Registro_Estados_Janeiro_por_Equipamento.xlsx`: 106 estados operacionais + `Classe`).

## Algoritmo
1. Sorteia `N_INICIAL = 100` linhas, de forma **estratificada**: as turmas entram na mesma proporção da base.
2. Para **cada turma**, calcula os **8 índices de CEQ** (Cp, Cpk, Pp, Ppk, p, np, c, u) nas linhas da turma, com as mesmas fórmulas do notebook 2 (limites LSL/USL = percentis 1 e 99). A amostra é mantida em ordem temporal, porque o σ de curto prazo do Cp/Cpk usa a amplitude móvel entre linhas consecutivas.
3. Mede a **reprodutibilidade** do top-X de cada índice em cada turma: divide as linhas da turma 10 vezes em duas metades aleatórias e calcula a fração média de atributos em comum entre o top-X de uma metade e o da outra.
4. Se a reprodutibilidade de algum índice em alguma turma ainda está melhorando, **retira 1 linha da base e coloca na amostra**, e volta ao passo 2.
5. Para quando a reprodutibilidade dos 8 índices **nas cinco turmas** deixa de melhorar (ganho < 0,03 em relação a uma amostra 20% menor e pelo menos 100 linhas menor, mantido por 50 iterações seguidas, com média móvel para tirar o ruído). Há um piso de 197 linhas na amostra (critério do IC do σ) e um teto de 50% da base, para garantir dados aos modelos.

## Saídas
- `amostra.xlsx`, com as abas `Amostra` (segue para o notebook 2, em ordem temporal), `Resultados_<turma>` (diagnóstico de suficiência de cada atributo, por turma), `Resumo`, `TopX_CEQ_Final`, `Reprodutibilidade` (índices de CEQ e métodos da literatura, por turma) e `Historico_Iteracoes`;
- `base_restante.xlsx`: a base original **sem** as linhas da amostra. Amostra + restante = original, sem repetição.

O arquivo original só é lido. Os ausentes são imputados pela mediana **da amostra** nos dois arquivos, para que nada da base restante vaze para a amostra. O sorteio é reprodutível (`SEMENTE`).

> **Coerência com o notebook 2**: Cp, Cpk, Pp e Ppk usam como limites de especificação os **percentis 1 e 99** de cada atributo. Se mudar lá (`PERCENTIL_LSL`/`PERCENTIL_USL`), mude também aqui.


### Selecionar o arquivo de entrada


In [ ]:
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)
arquivo_entrada = filedialog.askopenfilename(
    title="Selecione o arquivo de entrada (.xlsx)",
    filetypes=[("Arquivos Excel", "*.xlsx")],
)
root.destroy()
if not arquivo_entrada:
    raise ValueError("Nenhum arquivo foi selecionado.")
print(f"Arquivo selecionado (somente leitura, nunca é modificado): {arquivo_entrada}")


### Parâmetros


In [ ]:
# Parâmetros do notebook -- é só aqui que se altera.

CLASSES_INTERESSE = None   # None = todas as turmas (cada turma é a classe de referência do seu cenário);
                           # ou uma lista, ex.: ["A", "C"]
COLUNA_CLASSE = None      # None = última coluna da planilha

MODO = "amostragem"       # "amostragem" (padrão): começa com N_INICIAL linhas e move 1 linha
                          #     por vez da base para a amostra até o critério de parada;
                          #     gera a amostra (rankings) e a base restante (modelos)
                          # "auditoria": não sorteia nada, só relata a suficiência da base inteira

# ---- Critério de parada: reprodutibilidade do top-X dos 8 índices de CEQ ----
# A cada iteração, a classe de interesse da amostra é dividida N_DIVISOES vezes em
# duas metades aleatórias; a REPRODUTIBILIDADE de um índice é a fração média de
# sensores em comum entre o top-X de uma metade e o da outra (1.0 = as duas
# metades recomendam exatamente os mesmos sensores à empresa).
# A amostra para quando a reprodutibilidade de TODOS os 8 índices deixa de
# melhorar: ganho < GANHO_MIN em relação a uma amostra 20% menor (e no mínimo
# GAP_MIN linhas menor), mantido por PERSISTENCIA iterações seguidas (lógica de
# cotovelo, protegida contra paradas por ruído). Um índice que estagnou num patamar baixo não trava a parada -- o
# patamar dele fica registrado na aba Reprodutibilidade para a discussão.
TOP_X = 20                # tamanho da lista de sensores recomendada à empresa

# ---- Limites de especificação de Cp/Cpk/Pp/Ppk (TÊM de ser iguais aos do notebook 2) ----
# Sem limites de engenharia fornecidos pela empresa, LSL e USL são estimados pelos
# percentis do próprio sensor na classe de interesse: P1 e P99 = a faixa que
# contém 98% das leituras, ignorando 1% dos valores mais extremos de cada lado.
# (Antes: mínimo e máximo -- uma única leitura atípica mudava o limite, o ranking
# não era reprodutível e o Pp ficava idêntico ao AmplitudeSigma do grupo de controle.)
PERCENTIL_LSL = 1
PERCENTIL_USL = 99
N_DIVISOES = 10           # nº de divisões em metades por iteração
GANHO_MIN = 0.03          # ganho de reprodutibilidade considerado desprezível
SUAVIZACAO = 15           # média móvel (em iterações) para tirar o ruído das divisões
GAP_MIN = 100             # o ganho é medido contra uma amostra pelo menos 100 linhas menor
PERSISTENCIA = 50         # a estagnação precisa se manter por 50 iterações seguidas
N_MIN_AMOSTRA = 197       # piso: nº mínimo de linhas da AMOSTRA antes de poder parar
                          # (critério 1: IC do σ com largura <= 20%; mesma regra do SECOM)

# Diagnóstico (aba Resultados) -- não decide a parada
ALPHA = 0.05
ERRO_RELATIVO_MAX = 0.20
TOL_ESTABILIDADE = 0.10
N_PERMUTACOES = 20

N_INICIAL = 100           # linhas iniciais da amostra
INCREMENTO = 1            # linhas movidas da base para a amostra a cada iteração
FRACAO_MAXIMA = 0.50      # teto da amostra (fração da base) -- garante que sobrem dados
                          # (inclusive da classe minoritária) para treinar os modelos
ARQUIVO_AMOSTRA = "amostra.xlsx"
ARQUIVO_RESTANTE = "base_restante.xlsx"   # base original SEM as linhas da amostra -> modelos

SEMENTE = 42              # reprodutibilidade: mesma semente -> mesma amostra/mesma auditoria


### Funções


In [ ]:
import warnings
import numpy as np
import pandas as pd
from scipy import stats


def ler_base(caminho):
    """Lê o Excel decidindo se a 1ª linha é cabeçalho ou dado.

    Regra: lê sem cabeçalho e olha a 1ª linha. Ela é CABEÇALHO se tiver algum texto
    numa coluna cujo restante é numérico (ex.: UCI-SECOM: '0, 1, ..., 589, Pass/Fail',
    em que 'Pass/Fail' fica em cima de -1/1), ou se for quase toda texto.
    Caso contrário é DADO (SECOM crua, sem cabeçalho) e as colunas viram
    Attr_1..Attr_k + 'Classe' (última coluna = rótulo)."""
    bruto = pd.read_excel(caminho, header=None)
    if len(bruto) < 2:
        raise ValueError(f"'{caminho}' tem menos de 2 linhas.")
    linha0 = bruto.iloc[0]
    resto_num = bruto.iloc[1:].apply(lambda c: pd.to_numeric(c, errors="coerce").notna().mean() > 0.9)
    l0_texto = pd.to_numeric(linha0, errors="coerce").isna() & linha0.notna()
    tem_cabecalho = bool((l0_texto & resto_num).any() or l0_texto.mean() > 0.5)
    if tem_cabecalho:
        df = bruto.iloc[1:].reset_index(drop=True)
        def nome(v):  # 0.0 -> "0" (cabeçalhos numéricos lidos como float)
            return str(int(v)) if isinstance(v, float) and v.is_integer() else str(v).strip()
        df.columns = [nome(v) for v in linha0]
        for c in df.columns:  # colunas que eram só número voltam a ser numéricas
            conv = pd.to_numeric(df[c], errors="coerce")
            if conv.notna().sum() == df[c].notna().sum():
                df[c] = conv
        print("1ª linha reconhecida como cabeçalho.")
    else:
        df = bruto
        df.columns = [f"Attr_{i + 1}" for i in range(df.shape[1] - 1)] + ["Classe"]
        print("Arquivo sem cabeçalho: colunas nomeadas Attr_1..Attr_k, última = 'Classe'.")
    return df


def largura_relativa_ic_sigma(n, alpha=ALPHA):
    """Largura relativa (ls - li)/s do IC qui-quadrado do σ.
    Observação: o s se cancela -- a largura depende SÓ de n, não dos dados."""
    n = np.asarray(n, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        gl = n - 1
        li = np.sqrt(gl / stats.chi2.ppf(1 - alpha / 2, gl))
        ls = np.sqrt(gl / stats.chi2.ppf(alpha / 2, gl))
    return np.where(n >= 2, ls - li, np.nan)


def n_minimo_ic(erro_max=ERRO_RELATIVO_MAX, alpha=ALPHA):
    """Menor n tal que a largura relativa do IC do σ fique <= erro_max."""
    n = 3
    while largura_relativa_ic_sigma(n, alpha) > erro_max:
        n += 1
    return n


def trajetoria_media_sigma(valores):
    """Média e σ (ddof=1) acumulados para n = 1..N, vetorizado (sem loop)."""
    x = np.asarray(valores, dtype=float)
    n = np.arange(1, len(x) + 1)
    s1 = np.cumsum(x)
    s2 = np.cumsum(x * x)
    media = s1 / n
    with np.errstate(invalid="ignore", divide="ignore"):
        var = (s2 - n * media ** 2) / (n - 1)
    sigma = np.sqrt(np.clip(var, 0, None))
    sigma[0] = np.nan
    return media, sigma


def n_estabilizacao(valores, tol=TOL_ESTABILIDADE):
    """Menor n a partir do qual média e σ acumulados ficam SEMPRE dentro de
    ±tol·σ_final dos valores finais (σ relativo ao σ final; média em unidades de σ)."""
    media, sigma = trajetoria_media_sigma(valores)
    s_fin, m_fin = sigma[-1], media[-1]
    if not np.isfinite(s_fin) or s_fin == 0:
        return np.nan
    fora = (np.abs(sigma / s_fin - 1) > tol) | (np.abs(media - m_fin) / s_fin > tol)
    fora[0] = True
    idx_fora = np.flatnonzero(fora)
    return int(idx_fora[-1] + 2) if len(idx_fora) else 1  # +1 (índice->n) +1 (próximo n)


def n_estabilizacao_permutacoes(x, n_perm=N_PERMUTACOES, tol=TOL_ESTABILIDADE):
    """n de estabilização de média e σ em `n_perm` ordens aleatórias dos dados
    (vetorizado). Usa SEMPRE a mesma semente: para os mesmos dados, a resposta é
    sempre a mesma -- a decisão da amostragem e a da auditoria final coincidem."""
    n = len(x)
    P = np.random.default_rng(SEMENTE).permuted(np.tile(x, (n_perm, 1)), axis=1)
    k = np.arange(1, n + 1)
    media = np.cumsum(P, axis=1) / k
    with np.errstate(invalid="ignore", divide="ignore"):
        sigma = np.sqrt(np.clip((np.cumsum(P * P, axis=1) - k * media ** 2) / (k - 1), 0, None))
        s_fin, m_fin = sigma[:, -1:], media[:, -1:]
        fora = (np.abs(sigma / s_fin - 1) > tol) | (np.abs(media - m_fin) / s_fin > tol)
    fora[:, 0] = True
    return (n - 1 - np.argmax(fora[:, ::-1], axis=1)) + 2   # último n fora da faixa + 1


def aprovada_rapida(valores_classe, n_min):
    """Mesma decisão de auditar_coluna() (Suficiente ou Constante -> True), sem os
    testes de normalidade -- é a função chamada a cada iteração da amostragem."""
    x = valores_classe[~np.isnan(valores_classe)]
    n = len(x)
    if n < n_min:                      # critério 1 falha: nem precisa olhar estabilidade
        return False
    if np.std(x) == 0:
        return True
    return bool(np.percentile(n_estabilizacao_permutacoes(x), 95) <= n / 2)


def testes_normalidade(x):
    if len(x) < 8 or np.std(x) == 0:
        return {"Shapiro_p": np.nan, "JB_p": np.nan, "Normalidade_OK": False}
    amostra = x if len(x) <= 5000 else np.random.default_rng(SEMENTE).choice(x, 5000, replace=False)
    _, sh_p = stats.shapiro(amostra)
    _, jb_p = stats.jarque_bera(x)
    return {"Shapiro_p": sh_p, "JB_p": jb_p, "Normalidade_OK": bool(sh_p > 0.05 and jb_p > 0.05)}


def auditar_coluna(valores_classe, n_min):
    """Auditoria de UMA coluna, só com as linhas da classe de interesse."""
    x = valores_classe[~np.isnan(valores_classe)]
    n = len(x)
    linha = {"n_validos": n, "n_ausentes": int(len(valores_classe) - n)}
    if n < 2:
        linha.update({"media": np.nan, "sigma": np.nan, "IC_sigma_largura_rel": np.nan,
                      "Crit1_IC_OK": False, "n_estabilizacao_p95": np.nan,
                      "Folga_estabilidade": np.nan, "Crit2_Estavel_OK": False,
                      "Constante": False, "Status": "Sem dados suficientes"})
        return linha
    sigma = np.std(x, ddof=1)
    largura = float(largura_relativa_ic_sigma(n))
    linha.update({"media": np.mean(x), "sigma": sigma, "IC_sigma_largura_rel": largura,
                  "Crit1_IC_OK": bool(n >= n_min)})
    if sigma == 0:
        linha.update({"n_estabilizacao_p95": np.nan, "Folga_estabilidade": np.nan,
                      "Crit2_Estavel_OK": True, "Constante": True,
                      "Status": "Constante (σ=0) -- sem informação para CEQ"})
        linha.update(testes_normalidade(x))
        return linha
    n_est95 = float(np.percentile(n_estabilizacao_permutacoes(x), 95))
    # Estável = estabiliza (em 95% das ordens) usando no máximo metade dos dados disponíveis
    estavel = n_est95 <= n / 2
    linha.update({"n_estabilizacao_p95": n_est95, "Folga_estabilidade": n / n_est95,
                  "Crit2_Estavel_OK": bool(estavel), "Constante": False})
    linha.update(testes_normalidade(x))
    if linha["Crit1_IC_OK"] and estavel:
        linha["Status"] = "Suficiente"
    elif linha["Crit1_IC_OK"]:
        linha["Status"] = "IC OK, mas estimativa ainda instável"
    elif estavel:
        linha["Status"] = "Estável, mas IC do σ largo (n pequeno)"
    else:
        linha["Status"] = "Insuficiente"
    return linha


def separar_colunas(df, coluna_classe):
    """Colunas numéricas de atributo (exclui a classe e colunas de texto/data)."""
    return [c for c in df.select_dtypes(include=[np.number]).columns if c != coluna_classe]


INDICES_CEQ = ["Cp", "Cpk", "Pp", "Ppk", "p", "np", "c", "u"]


def indices_ceq_matriz(A):
    """Os 8 índices do notebook 2 (calcular_ceq + calcular_graficos_controle),
    vetorizados. A: matriz n x p da classe de interesse, JÁ imputada e em ordem
    temporal (o σ de curto prazo de Cp/Cpk usa a amplitude móvel entre linhas
    consecutivas). Limites LSL/USL = percentis PERCENTIL_LSL/PERCENTIL_USL de cada
    sensor. Colunas inválidas (constantes) recebem NaN, como no notebook 2."""
    n = A.shape[0]
    lsl, usl = np.percentile(A, [PERCENTIL_LSL, PERCENTIL_USL], axis=0)
    media = A.mean(0)
    s_longo = A.std(0)                                               # ddof=0
    s_curto = (np.abs(np.diff(A, axis=0)).mean(0) / 1.128) if n >= 2 else np.full(A.shape[1], np.nan)
    ok = (n >= 2) & (s_longo > 0) & (s_curto > 0) & np.isfinite(s_curto)
    with np.errstate(invalid="ignore", divide="ignore"):
        r = {
            "Cp": (usl - lsl) / (6 * s_curto),
            "Cpk": np.minimum(usl - media, media - lsl) / (3 * s_curto),
            "Pp": (usl - lsl) / (6 * s_longo),
            "Ppk": np.minimum(usl - media, media - lsl) / (3 * s_longo),
        }
    r = {k: np.where(ok, v, np.nan) for k, v in r.items()}
    ok_gc = (n >= 2) & (s_longo > 0)
    defeito = ((A < media - s_longo) | (A > media + s_longo)) & ok_gc   # fora de ±1σ
    d_amostra = defeito.sum(1)                                          # atributos fora, por linha
    n_def = defeito.sum(0)
    c = (defeito * d_amostra[:, None]).sum(0)
    r.update({"p": np.where(ok_gc, n_def / n, np.nan), "np": np.where(ok_gc, n_def, np.nan),
              "c": np.where(ok_gc, c, np.nan), "u": np.where(ok_gc, c / n, np.nan)})
    return r


def top_x(scores, validas, x):
    """Índices das x colunas de maior score (desempate pela ordem da coluna)."""
    sc = np.where(validas & np.isfinite(scores), scores, -np.inf)
    return frozenset(np.argsort(-sc, kind="mergesort")[:x].tolist())


def ordem_estratificada(df, coluna_classe, rng):
    """Ordem de sorteio em que cada classe aparece na proporção da base ao longo
    de toda a sequência -- qualquer prefixo (a amostra) e o restante (a base de
    modelagem) recebem a classe minoritária na mesma proporção."""
    chave = pd.Series(np.nan, index=df.index)
    for _, idx in df.groupby(coluna_classe).groups.items():
        idx = rng.permutation(np.asarray(idx))
        chave.loc[idx] = (np.arange(len(idx)) + rng.random(len(idx))) / len(idx)
    return chave.sort_values(kind="mergesort").index.to_numpy()


def matriz_classe_imputada(Xs, eh_int_s, ordem_tempo_s):
    """Reproduz o que o notebook 2 vai ver: imputa pela mediana da AMOSTRA (todas
    as classes, como no final deste notebook), filtra a classe de interesse e
    põe em ordem temporal. Retorna (matriz, colunas_validas)."""
    med = np.nanmedian(Xs, axis=0)
    validas = np.isfinite(med)                    # coluna 100% vazia na amostra -> descartada
    Xi = np.where(np.isnan(Xs), med, Xs)
    A = Xi[eh_int_s][np.argsort(ordem_tempo_s[eh_int_s], kind="mergesort")]
    return A, validas


def amostragem_adaptativa(df, coluna_classe, classes, rng):
    """Começa com N_INICIAL linhas (sorteio estratificado); a cada iteração move
    INCREMENTO linha(s) da base para a amostra, recalcula os 8 índices de CEQ
    (mesmas fórmulas do notebook 2) em N_DIVISOES pares de metades aleatórias e
    para quando a reprodutibilidade do top-X de todos os índices deixa de melhorar
    (ou no teto FRACAO_MAXIMA). Retorna (amostra, base_restante, info)."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return _amostragem(df, coluna_classe, classes, rng)


def reprodutibilidade(A, validas, calc_scores, n_div=N_DIVISOES, x=TOP_X, semente=SEMENTE):
    """Concordância média do top-x entre duas metades aleatórias das linhas de A
    (cada metade mantém a ordem temporal). calc_scores(A_sub) -> {nome: scores}."""
    r = np.random.default_rng(semente)
    n = len(A)
    soma = None
    for _ in range(n_div):
        p = r.permutation(n)
        sa, sb = calc_scores(A[np.sort(p[: n // 2])]), calc_scores(A[np.sort(p[n // 2:])])
        conc = {m: len(top_x(sa[m], validas, x) & top_x(sb[m], validas, x)) / x for m in sa}
        soma = conc if soma is None else {m: soma[m] + conc[m] for m in conc}
    return {m: v / n_div for m, v in soma.items()}


def _amostragem(df, coluna_classe, classes, rng):
    """Versão multiclasse do estudo de caso 1: a regra de parada é a mesma, mas a
    reprodutibilidade é medida em CADA turma (cada uma é a classe de referência do
    seu cenário) e a amostra só para quando as 8 x len(classes) séries estagnam."""
    cols = separar_colunas(df, coluna_classe)
    ordem = ordem_estratificada(df, coluna_classe, rng)
    X = df.loc[ordem, cols].to_numpy(dtype=float)
    rotulos = df.loc[ordem, coluna_classe].to_numpy()
    eh_cls = {c: (rotulos == c) for c in classes}
    pos_tempo = df.index.get_indexer(ordem)          # posição original (ordem temporal)
    n_max = max(N_INICIAL, int(np.floor(FRACAO_MAXIMA * len(ordem))))
    series = [f"{c}|{m}" for c in classes for m in INDICES_CEQ]

    hist = []                # uma linha por iteração (aba Historico_Iteracoes)
    k, iteracao, parou, seguidas = N_INICIAL, 0, False, 0
    while True:
        iteracao += 1
        rep = {}
        for c in classes:
            A, validas = matriz_classe_imputada(X[:k], eh_cls[c][:k], pos_tempo[:k])
            rep[c] = reprodutibilidade(A, validas, indices_ceq_matriz)
        linha = {"Iteracao": iteracao, "Linhas_amostra": k}
        linha.update({f"Linhas_classe_{c}": int(eh_cls[c][:k].sum()) for c in classes})
        linha.update({f"{c}|{m}": rep[c][m] for c in classes for m in INDICES_CEQ})
        hist.append(linha)
        h = pd.DataFrame(hist)
        suav = h[series].rolling(SUAVIZACAO, min_periods=SUAVIZACAO).mean()
        k_ref = min(int(round(0.8 * k)), k - GAP_MIN)  # 20% menor e pelo menos GAP_MIN linhas
        linha_ref = np.flatnonzero(h["Linhas_amostra"].to_numpy() <= k_ref)
        ganho = None
        if len(linha_ref) and not suav.iloc[linha_ref[-1]].isna().any() and not suav.iloc[-1].isna().any():
            ganho = suav.iloc[-1] - suav.iloc[linha_ref[-1]]
        estagnados = int((ganho < GANHO_MIN).sum()) if ganho is not None else 0
        seguidas = seguidas + 1 if (estagnados == len(series) and k >= N_MIN_AMOSTRA) else 0
        if iteracao == 1 or iteracao % 50 == 0:
            print(f"   iteração {iteracao}: {k} linhas -- {estagnados}/{len(series)} séries "
                  f"(turma x índice) já não melhoram")
        if seguidas >= PERSISTENCIA:
            print(f"Reprodutibilidade dos 8 índices nas {len(classes)} turmas sem melhora "
                  f"(ganho < {GANHO_MIN} em relação a {k_ref} linhas) por {PERSISTENCIA} iterações "
                  f"seguidas: parada com {k} linhas, iteração {iteracao}.")
            parou = True
            break
        if k + INCREMENTO > n_max:
            print(f"Teto de {FRACAO_MAXIMA:.0%} da base atingido ({k} linhas): "
                  f"{estagnados}/{len(series)} séries já tinham estagnado. Veja a aba Reprodutibilidade.")
            break
        k += INCREMENTO

    top_final, resumo_rep = [], []
    for c in classes:
        A, validas = matriz_classe_imputada(X[:k], eh_cls[c][:k], pos_tempo[:k])
        indices = indices_ceq_matriz(A)
        for m in INDICES_CEQ:
            sc = np.where(validas & np.isfinite(indices[m]), indices[m], -np.inf)
            for pos, j in enumerate(np.argsort(-sc, kind="mergesort")[:TOP_X], start=1):
                top_final.append({"Turma": c, "Indice": m, "Posicao": pos, "Atributo": cols[j], "Score": indices[m][j]})
        # Mesma medida, no mesmo ponto de parada, para métodos da literatura (turma c contra as demais)
        rep_lit = reprodutibilidade_literatura(X[:k], eh_cls[c][:k])
        ult = suav.iloc[-1]
        for m in INDICES_CEQ:
            v = float(ult[f"{c}|{m}"]) if not ult.isna().any() else float(rep[c][m])
            resumo_rep.append({"Turma": c, "Metodo": m, "Grupo": "Proposta (CEQ)", "Reprodutibilidade_topX": v})
        resumo_rep += [{"Turma": c, "Metodo": m, "Grupo": "Literatura", "Reprodutibilidade_topX": v}
                       for m, v in rep_lit.items()]

    amostra = df.loc[np.sort(ordem[:k])]          # ORDEM TEMPORAL (necessária para Cp/Cpk)
    restante = df.loc[np.sort(ordem[k:])]
    info = {"convergiu": parou, "iteracoes": iteracao, "historico": h,
            "top_final": pd.DataFrame(top_final), "reprodutibilidade": pd.DataFrame(resumo_rep)}
    return amostra, restante, info


def reprodutibilidade_literatura(Xs, y, n_div=N_DIVISOES, x=TOP_X, semente=SEMENTE):
    """Mesma medida de reprodutibilidade (metades aleatórias, estratificadas por
    classe) para os métodos rápidos da literatura -- comparação justa, no mesmo
    ponto de parada, sem regra diferente para cada grupo."""
    r = np.random.default_rng(semente)
    i1, i0 = np.flatnonzero(y), np.flatnonzero(~y)
    soma = None
    for _ in range(n_div):
        p1, p0 = r.permutation(i1), r.permutation(i0)
        a = np.concatenate([p1[: len(p1) // 2], p0[: len(p0) // 2]])
        b = np.concatenate([p1[len(p1) // 2:], p0[len(p0) // 2:]])
        (sa, ia), (sb, ib) = scores_literatura(Xs[a], y[a]), scores_literatura(Xs[b], y[b])
        conc = {}
        for m in sa:
            ta = set(ia[np.argsort(-sa[m], kind="mergesort")[:x]].tolist())
            tb = set(ib[np.argsort(-sb[m], kind="mergesort")[:x]].tolist())
            conc[m] = len(ta & tb) / x
        soma = conc if soma is None else {m: soma[m] + conc[m] for m in conc}
    return {m: v / n_div for m, v in soma.items()}


def scores_literatura(Xs, y):
    """Versões rápidas e determinísticas de 7 métodos da literatura do notebook 2
    (mesma convenção: maior = melhor). ReliefF, MI/IG/GainRatio/SU, MIC, Gini,
    SVD, Laplacian e SPEC ficam de fora por custo ou aleatoriedade."""
    from sklearn.feature_selection import f_classif, chi2
    from scipy.stats import rankdata
    med = np.nanmedian(Xs, axis=0)
    Xi = np.where(np.isnan(Xs), med, Xs)
    validas = np.isfinite(med) & (Xi.std(0) > 0)
    Xi = Xi[:, validas]
    g1, g0 = Xi[y], Xi[~y]
    yc = y.astype(float) - y.mean()
    def corr_abs(M):
        Mc = M - M.mean(0)
        return np.abs((Mc * yc[:, None]).sum(0) / (np.sqrt((Mc ** 2).sum(0) * (yc ** 2).sum()) + 1e-12))
    mn = (Xi - Xi.min(0)) / (Xi.max(0) - Xi.min(0))
    m1, m0, v1, v0 = g1.mean(0), g0.mean(0), g1.var(0, ddof=1), g0.var(0, ddof=1)
    mg = Xi.mean(0)
    out = {
        "ANOVA": f_classif(Xi, y)[0],
        "Pearson": corr_abs(Xi),
        "Spearman": corr_abs(np.apply_along_axis(rankdata, 0, Xi)),
        "Chi2": chi2(mn, y)[0],
        "TScore": np.abs(m1 - m0) / np.sqrt(v1 / len(g1) + v0 / len(g0) + 1e-12),
        "FisherScore": (len(g1) * (m1 - mg) ** 2 + len(g0) * (m0 - mg) ** 2)
                       / (len(g1) * v1 + len(g0) * v0 + 1e-9),
    }
    s1, s0 = np.sort(g1, 0), np.sort(g0, 0)
    ks = np.empty(Xi.shape[1])
    for j in range(Xi.shape[1]):
        grade = np.concatenate([s1[:, j], s0[:, j]])
        ks[j] = np.max(np.abs(np.searchsorted(s1[:, j], grade, "right") / len(g1)
                              - np.searchsorted(s0[:, j], grade, "right") / len(g0)))
    out["KS"] = ks
    idx_validas = np.flatnonzero(validas)
    return {m: np.nan_to_num(v, nan=-np.inf) for m, v in out.items()}, idx_validas


def imputar_e_limpar(df, cols, outro=None):
    """Imputação por mediana DA AMOSTRA (só colunas numéricas de atributo) e remoção
    de colunas sem nenhuma leitura válida na amostra. Se `outro` (base restante) for
    passado, recebe as MESMAS remoções e as MESMAS medianas da amostra (sem usar
    informação da base restante na amostra)."""
    df = df.copy()
    vazias = [c for c in cols if df[c].isna().all()]
    if vazias:
        df = df.drop(columns=vazias)
        print(f"{len(vazias)} coluna(s) sem nenhuma leitura válida removida(s): {vazias}")
    cols = [c for c in cols if c not in vazias]
    n_aus = int(df[cols].isna().sum().sum())
    medianas = df[cols].median()
    df[cols] = df[cols].fillna(medianas)
    print(f"Imputação por mediana: {n_aus} valor(es) ausente(s) em {len(cols)} coluna(s).")
    if outro is not None:
        outro = outro.drop(columns=vazias).copy()
        outro[cols] = outro[cols].fillna(medianas)
    return df, outro


### Execução

A execução leva alguns minutos na base dos caminhões (cinco turmas, 106 atributos). O progresso aparece a cada 50 iterações.

Nas abas `Resultados_<turma>` (diagnóstico de cada atributo, que não decide a parada), a coluna `Status` vale:
- `Suficiente`: n ≥ 197 na turma, com média e σ estáveis;
- `IC OK, mas estimativa ainda instável`: σ dominado por outliers;
- `Insuficiente`: menos de 197 leituras válidas na turma (esperado: cada turma tem só uma fração da amostra);
- `Constante (σ=0)`: índices de capacidade indefinidos.


In [ ]:
def executar(caminho_entrada):
    rng = np.random.default_rng(SEMENTE)
    df = ler_base(caminho_entrada)            # o arquivo original só é LIDO, nunca regravado
    col_classe = COLUNA_CLASSE if COLUNA_CLASSE is not None else df.columns[-1]
    existentes = sorted(df[col_classe].dropna().unique())
    classes = existentes if CLASSES_INTERESSE is None else list(CLASSES_INTERESSE)
    faltando = [c for c in classes if c not in existentes]
    if faltando:
        raise ValueError(f"Classe(s) {faltando} não encontrada(s) em '{col_classe}'. Valores existentes: {existentes}")
    print(f"Base original: {len(df)} linhas de dados (fora o cabeçalho), "
          f"{df.shape[1]} colunas, classe em '{col_classe}': {existentes}.")

    n_min = n_minimo_ic()
    print(f"Critério 1 exige n >= {n_min} leituras válidas por coluna "
          f"(IC do σ com largura <= {ERRO_RELATIVO_MAX:.0%}).")

    info = None
    if MODO == "amostragem":
        amostra, restante, info = amostragem_adaptativa(df, col_classe, classes, rng)
    elif MODO == "auditoria":
        amostra, restante = df, None
        print(f"Modo AUDITORIA: base inteira ({len(df)} linhas), nada é sorteado.")
    else:
        raise ValueError("MODO deve ser 'amostragem' ou 'auditoria'.")

    cols = separar_colunas(amostra, col_classe)
    # Auditoria da amostra final, uma por turma -- ANTES da imputação (só leituras reais)
    resultados = {}
    for c in classes:
        valores_classe = amostra.loc[amostra[col_classe] == c, cols]
        r = pd.DataFrame({a: auditar_coluna(valores_classe[a].to_numpy(dtype=float), n_min) for a in cols}).T
        r.index.name = "Coluna"
        resultados[c] = r

    contagem = pd.DataFrame({
        "Amostra": amostra[col_classe].value_counts(),
        "Base restante": (restante[col_classe].value_counts() if restante is not None else 0),
        "Base original": df[col_classe].value_counts(),
    }).fillna(0).astype(int).rename_axis("Classe").reset_index()

    resumo = pd.DataFrame({
        "Parâmetro": ["Arquivo de entrada", "Modo", "Coluna de classe", "Turmas (classes de referência)",
                      "Linhas na amostra", "Linhas na base restante",
                      "n mínimo (critério 1)", "Erro relativo máx. do σ",
                      "Tolerância de estabilidade", "Semente", "Colunas avaliadas",
                      "Critério de parada", "Convergiu?", "Iterações"],
        "Valor": [str(caminho_entrada), MODO, col_classe, ", ".join(map(str, classes)),
                  len(amostra), (len(restante) if restante is not None else 0),
                  n_min, ERRO_RELATIVO_MAX, TOL_ESTABILIDADE, SEMENTE, len(cols),
                  (f"reprodutibilidade do top-{TOP_X} dos 8 índices CEQ, nas {len(classes)} turmas, parou de "
                   f"melhorar (ganho < {GANHO_MIN} vs. amostra 20% e >= {GAP_MIN} linhas menor, por "
                   f"{PERSISTENCIA} iterações), piso de {N_MIN_AMOSTRA} linhas na amostra, "
                   f"teto de {FRACAO_MAXIMA:.0%} da base") if info else "-",
                  (("Sim" if info["convergiu"] else "NÃO (teto atingido)") if info else "-"),
                  (info["iteracoes"] if info else "-")],
    })

    amostra_out, restante_out = imputar_e_limpar(amostra, cols, restante)
    with pd.ExcelWriter(ARQUIVO_AMOSTRA) as w:
        amostra_out.to_excel(w, sheet_name="Amostra", index=False)
        for c, r in resultados.items():
            r.to_excel(w, sheet_name=f"Resultados_{c}")
        resumo.to_excel(w, sheet_name="Resumo", index=False)
        contagem.to_excel(w, sheet_name="Resumo", index=False, startrow=len(resumo) + 2)
        if info:
            info["top_final"].to_excel(w, sheet_name="TopX_CEQ_Final", index=False)
            info["reprodutibilidade"].to_excel(w, sheet_name="Reprodutibilidade", index=False)
            info["historico"].to_excel(w, sheet_name="Historico_Iteracoes", index=False)
    if restante_out is not None:
        restante_out.to_excel(ARQUIVO_RESTANTE, sheet_name="Base_Restante", index=False)

    print("\n" + resumo.to_string(index=False))
    print("\nLinhas por classe:\n" + contagem.to_string(index=False))
    if info:
        print(f"\nReprodutibilidade do top-{TOP_X} na amostra final (1.0 = duas metades recomendam os mesmos atributos):")
        print(info["reprodutibilidade"].pivot_table(index="Metodo", columns="Turma", values="Reprodutibilidade_topX").round(2).to_string())
    print(f"\n{ARQUIVO_AMOSTRA}" + (f" e {ARQUIVO_RESTANTE}" if restante_out is not None else "") + " salvos.")
    return amostra_out, restante_out, resultados


if __name__ == "__main__":
    amostra, base_restante, resultados = executar(arquivo_entrada)
